# Serverless Inference

Deployed to a **serverless** endpoint.

No instance runs continuously, SageMaker provisions compute on request and scales it back down when idle. 

In [ ]:
import boto3
import sagemaker
from sagemaker.estimator import Estimator

# NOTE: Provide the region and training job name for attaching to the existing training job
REGION = ""
TRAINING_JOB_NAME = ""

boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)
estimator = Estimator.attach(TRAINING_JOB_NAME, sagemaker_session=sm_session)

## 1. Deploy with a `ServerlessInferenceConfig`

- `memory_size_in_mb` — must be one of a fixed set (1024, 2048, 3072, 4096, 5120, 6144);
  also determines the vCPUs allocated. Undersizing causes out-of-memory errors, oversizing
  wastes nothing extra since billing is per-request, not per-provisioned-memory.
- `max_concurrency` — the ceiling on simultaneous in-flight requests before throttling.

In [ ]:
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5,
)


# For serverless deployment send serverless_inference_config and for realtime send instance_type and initial_instance_count

serverless_predictor = estimator.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="fraud-detection-serverless",
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)
print(f"Endpoint: {serverless_predictor.endpoint_name}")

## 2. Invoke

Same request shape as Method 1.

The client-side code for invoking a serverless endpoint is identical to a real-time one. 
Only the deploy-time config differs. 

**NOTE** : The **first** call below may take noticeably longer than the rest (cold start).

In [ ]:
# METHOD 1

import time

sample_row = ['6854.53', '0', '0', '0', '1', '0']

start = time.time()
result = serverless_predictor.predict(sample_row)
print(f"First call ({time.time() - start:.2f}s): {result[0][0]}")

start = time.time()
result = serverless_predictor.predict(sample_row)
print(f"Second call ({time.time() - start:.2f}s): {result[0][0]}")

In [ ]:
# METHOD 2

csv_body = ",".join(str(v) for v in sample_row)

runtime = boto_session.client("sagemaker-runtime")

response = runtime.invoke_endpoint(
    EndpointName=serverless_predictor.endpoint_name, # Name of deployed endpoint
    ContentType="text/csv", #type of input
    Body=csv_body, # actual input
)

In [ ]:
response["Body"].read().decode()

## 3. Cleanup

Serverless endpoints don't bill while idle, but still delete them when done — this also
removes the endpoint config, keeping the account tidy for the next module.

In [ ]:
serverless_predictor.delete_endpoint()
print("Endpoint deleted.")